# NB-13 — does the negation advantage depend on data volume?

**Run this in a second Kaggle session, in parallel with NB-12.** They share no inputs, no outputs
and no eval set, so they cannot interfere.

## The gap this fills

RQ2 says deleting the Bengali negation particle costs **9.34$\times$ less** perturbation than
retargeting to an unrelated sentence — measured on 38 hand-validated utterances, **at the 100-hour
final model only.** T1 has never been run at any other rung.

So a reviewer can ask: *is the linguistic vulnerability a property of the language, or only of a
well-trained model?* Nobody knows, including us.

This measures T1 and T5 on the **1-hour** model, same 38 utterances, same hand-built target pairs,
and compares the ratio against 9.34$\times$.

| outcome | what it means |
|---|---|
| ratio stays near 9$\times$ | the negation asymmetry is **scale-invariant** — a property of Bengali morphology, not of model quality. Strengthens RQ2 considerably. |
| ratio shrinks | the advantage needs a competent model; it is partly an artefact of accuracy |
| ratio grows | low-data models are *even more* exposed to morphology-targeted edits — the thesis's own argument, sharpened |

Every outcome is reportable and it is a genuinely new result, not a re-run.

## Why this is the only sensible T1 run

**T1 cannot use the 100-utterance T5 set.** Only **3 of those 100** utterances carry a standalone negation particle and **none of the
original 25** do, so a paired comparison there is impossible; the targets file has no T1 column
either. T1 lives on NB-03's
38-utterance hand-validated spine and nowhere else. Mixing the two sets is the mistake that cost
four runs on 2026-09-26; the config cell below asserts against it.

## Cost

~3 GPU-h for both conditions. T1 is cheap — every T1 attack succeeded at the 100 h rung, so the
bisection converges fast and censoring is low. T5 at the 1 h rung is the slower half.

## Attach

`ckpt-xlsr-1h`, `bn-asr-splits`, `bn-attack-targets` (NB-03's corpus — **required**, this run is
meaningless without the hand-validated pairs), and `bengaliai-speech`. GPU T4 x2, internet off.

## 1 · Configuration

In [ ]:
NOTEBOOK = "NB-13-t1-vs-t5-low-rung"

RUNG        = 1
CHECKPOINT  = "final"
CKPT_DATASET = "ckpt-xlsr-1h"     # the ORIGINAL 1 h run, the one RQ1's pair 1 uses
EDIT_TYPES  = ["T1", "T5"]        # both, on the SAME 38 utterances -> a paired ratio
EVAL_SET    = "nb03"              # attack_eval_t1.csv. NOT the 100-utt T5 set.

BATCH_SIZE     = 8
USE_AMP        = True
FIXED_EPS      = None
PROBE_STEPS    = 400
CONFIRM_STEPS  = 800
STALL_PATIENCE = 100
CHECK_EVERY    = 50
LR             = 1e-4
SEED           = 1337
WALL_HOURS     = 8.0
VALIDATE_BATCHING = False

# T1 targets are the source minus one word, so they land at the BOTTOM of the T5
# bracket. The 2026-09-25 probe returned 1.048e-4 against an EPS_LO of 1.0e-4:
# every value pinned to the floor, an upper bound rather than a measurement.
# T1 needs one more decade of headroom and one more bisection round to keep the
# resolution. This is exactly what the published RQ2 run used -- do not change it,
# or the ratio stops being comparable to 9.34x.
def bracket_for(edit):
    return ((1e-5, 2e-3), 6) if edit == "T1" else ((1e-4, 2e-3), 5)

assert EVAL_SET == "nb03", (
    "T1 requires NB-03's 38-utterance corpus. Only 3 of the 100-utterance T5 set "
    "carry a standalone negation particle, none of them in the original 25, so a "
    "paired T1-vs-T5 comparison is impossible there.")
assert set(EDIT_TYPES) == {"T1", "T5"}, (
    "Both edit types are required: the result is a PAIRED ratio on the same audio, "
    "not two independent numbers.")

print(f"NB-13 | rung {RUNG} {CHECKPOINT} | {EDIT_TYPES} on the 38-utt NB-03 spine")
print("reference, same corpus, 100 h final: T1 100% (38/38), T5 36.8% (14/38), ratio 9.34x")

## 2 · Boilerplate

In [ ]:
import os, hashlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # attacks optimise the INPUT: never DataParallel
import sys, json, time, random, re, glob, unicodedata
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
torch.backends.cudnn.benchmark = True

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK, TEMP, DEVICE = "/kaggle/working", "/kaggle/temp", "cuda"
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"

# PROV holds only what is defined AT THIS POINT. In NB-05 the rung, edit type
# and bracket were module-level constants, so they could be read here; in a
# queue runner they are per-condition and are not bound yet. Reading them here
# is a NameError that fires three cells before anything expensive starts. The
# run-specific fields are added by the summary cell at the end.
PROV = {"notebook": NOTEBOOK, "seed": SEED,
        "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
print(json.dumps(PROV, indent=2))

In [ ]:
# See NB-01 for the full explanation: pinning an old `accelerate`/`datasets`
# silently downgrades numpy mid-session and breaks pandas much later with
# "No module named 'numpy.rec'". Pin numpy to what is already loaded.
# `datasets` is not imported by this notebook, so it is not installed.
import numpy, subprocess, sys
NP_BEFORE = numpy.__version__

!pip install -q transformers==4.44.2 librosa soundfile jiwer "numpy=={NP_BEFORE}" 2>&1 | tail -2

NP_DISK = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
assert NP_BEFORE == NP_DISK, f"numpy moved {NP_BEFORE} -> {NP_DISK}; relax the pin that forced it."
print(f"numpy {NP_BEFORE} stable")

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import transformers, librosa
print("transformers", transformers.__version__)
PROV["transformers"] = transformers.__version__
PROV["numpy"] = NP_BEFORE


## 3 · Model, audio, and the hand-validated target pairs

In [ ]:
inputs = sorted({p for pat in ("/kaggle/input/*", "/kaggle/input/*/*",
                               "/kaggle/input/*/*/*")
                 for p in glob.glob(pat) if os.path.isdir(p)})

cands = [p for p in inputs if re.search(rf"{re.escape(CKPT_DATASET)}$", os.path.basename(p))]
assert len(cands) == 1, (f"need exactly one input matching '{CKPT_DATASET}', got "
                         f"{[os.path.basename(c) for c in cands]}")
CKPT = cands[0]
MODEL_DIR = CKPT if os.path.exists(f"{CKPT}/config.json") else f"{CKPT}/final"
assert os.path.exists(f"{MODEL_DIR}/config.json"), f"no config.json under {MODEL_DIR}"

processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).float().to(DEVICE).eval()
for p in model.parameters(): p.requires_grad_(False)
model.config.apply_spec_augment = False
DO_NORMALIZE = bool(getattr(processor.feature_extractor, "do_normalize", True))
BLANK_ID = model.config.pad_token_id
print("checkpoint:", MODEL_DIR)
print(f"do_normalize {DO_NORMALIZE} | blank {BLANK_ID} | vocab {model.config.vocab_size}")

SPLITS = next((p for p in inputs if "splits" in p.lower()), None)
assert SPLITS, "attach bn-asr-splits"
EV_CSV = next((f"{p}/attack_eval_t1.csv" for p in inputs
               if os.path.exists(f"{p}/attack_eval_t1.csv")), None)
assert EV_CSV, ("attack_eval_t1.csv not found. Attach bn-attack-targets (NB-03's output). "
                f"Visible: {[os.path.basename(p) for p in inputs]}")
TGT_CSV = next((f"{p}/bn_attack_targets.csv" for p in inputs
                if os.path.exists(f"{p}/bn_attack_targets.csv")), None)
assert TGT_CSV, "bn_attack_targets.csv not found -- attach bn-attack-targets"
print("eval set:", EV_CSV)
print("targets :", TGT_CSV)

ZW = dict.fromkeys(map(ord, "‌‍"), None)
def norm_bn(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).translate(ZW)).strip()

def resolve_audio(df):
    p0 = str(df["path"].iloc[0])
    if os.path.exists(p0): return df
    stem, ext = os.path.basename(os.path.dirname(p0)), os.path.splitext(p0)[1]
    c = [d for pat in (f"/kaggle/input/*/{stem}", f"/kaggle/input/*/*/{stem}",
                       f"/kaggle/input/*/*/*/{stem}")
         for d in glob.glob(pat) if os.path.isdir(d)]
    assert c, f"audio dir '{stem}' not found -- attach bengaliai-speech"
    df = df.copy(); df["path"] = c[0] + "/" + df["utt_id"].astype(str) + ext
    return df

base = resolve_audio(pd.read_csv(EV_CSV))
tg_all = pd.read_csv(TGT_CSV)

# Build one frame per edit type, then INTERSECT on utt_id. The ratio is only a
# paired statistic if both edit types run on exactly the same utterances; taking
# each edit type's own set independently would silently compare different audio.
EV = {}
for e in EDIT_TYPES:
    t = tg_all[tg_all.edit_type == e]
    d = base.merge(t[["source_utterance_id", "target_text"]],
                   left_on="utt_id", right_on="source_utterance_id", how="inner")
    d["target_text"] = d.target_text.map(norm_bn)
    EV[e] = d
    print(f"  {e}: {len(d)} pairs from the hand-built corpus")

shared = set.intersection(*(set(d.utt_id) for d in EV.values()))
for e in EDIT_TYPES:
    EV[e] = EV[e][EV[e].utt_id.isin(shared)].sort_values("utt_id").reset_index(drop=True)
assert len(shared) > 0, "no utterance carries BOTH a T1 and a T5 target"
assert all(list(EV[e].utt_id) == sorted(shared) for e in EDIT_TYPES), "utt_id order mismatch"
N_SHARED = len(shared)
print(f"\n{N_SHARED} utterances carry BOTH target types -- this is the paired set")
if N_SHARED != 38:
    print(f"  NOTE: the published RQ2 result used 38. You have {N_SHARED}. Say so in RUNLOG.")
for c in ("path", "text", "target_text", "duration"):
    for e in EDIT_TYPES:
        assert c in EV[e].columns, f"{e} frame is missing {c}"

PROV.update(notebook="NB-13-t1-vs-t5-low-rung", rung=RUNG, checkpoint=CHECKPOINT,
            ckpt_dataset=CKPT_DATASET, eval_set=EVAL_SET, model_dir=MODEL_DIR,
            attack_eval_csv=os.path.basename(EV_CSV), n_shared=int(N_SHARED),
            edit_types=EDIT_TYPES, seed=SEED)

## 4 · Front end, loss, PGD, bisection

Byte-identical to NB-05.

In [ ]:
SR = 16000

def load_batch(rows):
    """Length-bucketed batch. Returns padded waveforms + a validity mask."""
    wavs = []
    for _, r in rows.iterrows():
        a, _ = librosa.load(r["path"], sr=SR, mono=True)
        a = np.asarray(a, np.float32)
        pk = np.abs(a).max()
        wavs.append(a / max(pk, 1.0))
    T = max(len(w) for w in wavs)
    x = torch.zeros(len(wavs), T, dtype=torch.float32)
    m = torch.zeros(len(wavs), T, dtype=torch.float32)
    for i, w in enumerate(wavs):
        x[i, :len(w)] = torch.from_numpy(w); m[i, :len(w)] = 1.0
    return x.to(DEVICE), m.to(DEVICE)


def frontend_b(x, mask):
    """Masked zero-mean/unit-variance. Computing stats over the PADDING would
    corrupt short utterances in a mixed-length batch."""
    if DO_NORMALIZE:
        n = mask.sum(-1, keepdim=True).clamp(min=1)
        mu = (x * mask).sum(-1, keepdim=True) / n
        var = (((x - mu) * mask) ** 2).sum(-1, keepdim=True) / n
        x = (x - mu) / torch.sqrt(var + 1e-7)
        x = x * mask
    return x


def logits_b(x, mask):
    return model(input_values=frontend_b(x, mask),
                 attention_mask=mask.long()).logits


def ctc_per_sample(x, mask, tgt_pad, tgt_len):
    """Per-sample CTC loss -> lets us mask out finished samples."""
    lg = logits_b(x, mask)
    logp = F.log_softmax(lg.float(), -1).transpose(0, 1)          # (L,B,V)
    in_len = model._get_feat_extract_output_lengths(mask.sum(-1).long()).long()
    in_len = in_len.clamp(max=logp.shape[0])
    return F.ctc_loss(logp, tgt_pad, in_len, tgt_len,
                      blank=BLANK_ID, zero_infinity=True, reduction="none")


def decode_b(x, mask):
    with torch.no_grad():
        ids = torch.argmax(logits_b(x, mask), -1)
    return [norm_bn(s) for s in processor.batch_decode(ids)]


def int16_rt(x):
    return torch.clamp((x * 32767.0).round(), -32768.0, 32767.0) / 32767.0

def snr_db_b(clean, adv, mask):
    n = (adv - clean) * mask
    s = clean * mask
    return (10 * torch.log10(s.pow(2).sum(-1) / (n.pow(2).sum(-1) + 1e-12))).cpu().numpy()

In [ ]:
def pgd_batch(x, mask, tgt_pad, tgt_len, targets, eps_vec,
              steps, delta_init=None, lr=LR):
    """eps_vec: (B,) per-sample L-inf budgets.
       Returns (success bool array, frozen deltas, steps_used)."""
    B = x.shape[0]
    eps_col = eps_vec.view(-1, 1)
    delta = (torch.zeros_like(x) if delta_init is None
             else torch.max(torch.min(delta_init.clone(), eps_col), -eps_col))
    delta.requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    active = torch.ones(B, dtype=torch.bool, device=DEVICE)
    frozen = torch.zeros_like(x)
    success = np.zeros(B, dtype=bool)
    best_loss = torch.full((B,), float("inf"), device=DEVICE)
    stall = torch.zeros(B, dtype=torch.long, device=DEVICE)

    for step in range(steps):
        if not active.any():
            break
        x_adv = torch.clamp(x + delta, -1, 1) * mask
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            per = ctc_per_sample(x_adv, mask, tgt_pad, tgt_len)
        loss = (per * active.float()).sum() / active.float().sum().clamp(min=1)

        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if delta.grad is not None:
            delta.grad[~active] = 0          # finished samples stop moving
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            delta.data = torch.max(torch.min(delta.data, eps_col), -eps_col)

        with torch.no_grad():                # stall detection
            improved = per < best_loss - 1e-3
            best_loss = torch.minimum(best_loss, per.detach())
            stall = torch.where(improved, torch.zeros_like(stall), stall + 1)
            give_up = active & (stall > STALL_PATIENCE)
            active = active & ~give_up

        if (step + 1) % CHECK_EVERY == 0:
            hyp = decode_b(int16_rt(torch.clamp(x + delta, -1, 1)) * mask, mask)
            with torch.no_grad():
                for i in range(B):
                    if active[i] and hyp[i] == targets[i]:
                        frozen[i] = delta[i].detach()
                        success[i] = True
                        active[i] = False
    return success, frozen, step + 1

In [ ]:
def eps50_batch(x, mask, tgt_pad, tgt_len, targets, verbose=True):
    """Per-sample geometric bisection for the smallest budget that still works.

    ---------------------------------------------------------------------------
    BUG FIXED 2026-09-22 (found in the rung-1 T5 run, 40% censoring).

    The previous version ended with

        ok, best, _ = pgd_batch(..., hi, CONFIRM_STEPS, delta_init=best)

    which REASSIGNED `best` wholesale to pgd_batch's return value. That return
    is a `frozen` buffer holding zeros for any sample that did not succeed on
    that particular pass -- so one unlucky final pass wiped out a delta that had
    already been verified at a larger budget. Those samples were then written
    out as censored, with SNR 70-86 dB (the signature of delta == 0) even though
    the ceiling probe had already proved them attackable.

    In the rung-1 run that discarded ~6 of 25 real measurements. Worse, it does
    so at a RATE THAT VARIES BY MODEL, so comparing median eps50 across rungs
    would have been comparing differently-biased subsamples -- straight into
    RQ1's headline.

    Fix: keep a running `best_eps` / `best_delta` that is only ever updated on a
    VERIFIED success. Nothing overwrites a confirmed result. Censoring now means
    exactly one thing: the attack failed at every budget tried, ceiling included.

    Side benefit: eps50 becomes a guaranteed UPPER BOUND on the true minimum.
    Optimiser noise can only make it conservative, never wrong in an
    unpredictable direction -- which is the property you want in a metric you
    are about to compare across four models.
    ---------------------------------------------------------------------------
    """
    B = x.shape[0]
    lo = torch.full((B,), EPS_LO, device=DEVICE)
    hi = torch.full((B,), EPS_HI, device=DEVICE)
    best_eps   = torch.full((B,), float("nan"), device=DEVICE)   # verified only
    best_delta = torch.zeros_like(x)

    def record(okt, eps_vec, d):
        """Adopt a result only when it is both successful and better."""
        nonlocal best_eps, best_delta
        better = okt & (torch.isnan(best_eps) | (eps_vec < best_eps))
        best_eps   = torch.where(better, eps_vec, best_eps)
        best_delta = torch.where(better.view(-1, 1), d, best_delta)

    # Ceiling probe. Failure here is genuine right-censoring.
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, hi, CONFIRM_STEPS)
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, hi, d)
    feasible = ok.copy()
    if verbose:
        print(f"  ceiling probe eps={EPS_HI:.1e}: {feasible.sum()}/{B} feasible, "
              f"{(~feasible).sum()} censored")

    for rnd in range(BISECT_ROUNDS):
        mid = torch.sqrt(lo * hi)
        ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                             PROBE_STEPS, delta_init=best_delta)
        ok = ok & feasible
        okt = torch.tensor(ok, device=DEVICE)
        hi = torch.where(okt, mid, hi)
        lo = torch.where(okt, lo, mid)
        record(okt, mid, d)
        if verbose:
            print(f"  round {rnd+1}/{BISECT_ROUNDS}: {ok.sum()}/{B} succeeded "
                  f"| median eps {mid.median().item():.3e}")

    # Budget-consistency round: one more bisection at the FULL step budget, so
    # the reported boundary is measured at the budget eps50 is defined at rather
    # than at the cheaper probe budget. See RUNLOG for why this matters.
    #
    # Skipped in FIXED_EPS mode: there is no bracket to refine. The ceiling probe
    # already ran at CONFIRM_STEPS, so the single reported budget is measured at
    # the full budget and needs no second pass.
    if BISECT_ROUNDS == 0:
        eps       = best_eps.cpu().numpy()
        confirmed = ~np.isnan(eps)
        if verbose:
            print(f"  -> FIXED_EPS {EPS_HI:.1e}: {confirmed.sum()}/{B} succeeded, "
                  f"{(~confirmed).sum()} resisted")
        return eps, np.ones(B), confirmed, best_delta

    mid = torch.sqrt(lo * hi)
    ok, d, _ = pgd_batch(x, mask, tgt_pad, tgt_len, targets, mid,
                         CONFIRM_STEPS, delta_init=best_delta)
    ok = ok & feasible
    okt = torch.tensor(ok, device=DEVICE)
    record(okt, mid, d)
    if verbose:
        print(f"  budget-consistency round @ {CONFIRM_STEPS} steps: {ok.sum()}/{B}")

    eps       = best_eps.cpu().numpy()
    confirmed = ~np.isnan(eps)                 # censored == never succeeded
    width     = (hi / lo).cpu().numpy()
    if verbose:
        print(f"  -> {confirmed.sum()}/{B} measured, {(~confirmed).sum()} censored")
    return eps, width, confirmed, best_delta


## 5 · Run both edit types

In [ ]:
SESSION_START = time.time()
DEADLINE = SESSION_START + WALL_HOURS * 3600
summary = []

for e in EDIT_TYPES:
    (EPS_LO, EPS_HI), BISECT_ROUNDS = bracket_for(e)
    EDIT_TYPE = e
    ev = EV[e].sort_values("duration").reset_index(drop=True)
    OUT = f"{WORK}/eps50_rung{RUNG}_{CHECKPOINT}_{e}_n{N_SHARED}.csv"
    print("\n" + "=" * 74)
    print(f"{e}   bracket [{EPS_LO:.0e}, {EPS_HI:.0e}]   {BISECT_ROUNDS} rounds   "
          f"{len(ev)} utterances")
    print("=" * 74)

    results = []
    srcs = ([OUT] if os.path.exists(OUT) else []) + \
           sorted(glob.glob(f"/kaggle/input/*/{os.path.basename(OUT)}")) + \
           sorted(glob.glob(f"/kaggle/input/*/*/{os.path.basename(OUT)}"))
    seen = set()
    for p in srcs:
        for r in pd.read_csv(p).to_dict("records"):
            if r["utt_id"] not in seen: seen.add(r["utt_id"]); results.append(r)
    if results: print(f"  resuming: {len(results)} already measured")
    todo = ev[~ev.utt_id.isin(seen)].reset_index(drop=True)
    batches = [todo.iloc[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]

    for bi, sub in enumerate(batches):
        if time.time() > DEADLINE:
            print("  *** wall clock reached ***"); break
        xb, mb = load_batch(sub)
        tp = [torch.tensor(processor.tokenizer(t).input_ids, device=DEVICE)
              for t in sub.target_text]
        tl = torch.tensor([len(t) for t in tp], device=DEVICE)
        tpad = torch.nn.utils.rnn.pad_sequence(tp, batch_first=True, padding_value=0)
        t0 = time.time()
        eps, width, conf, best = eps50_batch(xb, mb, tpad, tl, list(sub.target_text),
                                             verbose=False)
        dt = time.time() - t0
        x_adv = int16_rt(torch.clamp(xb + best, -1, 1)) * mb
        snrs = snr_db_b(xb, x_adv, mb); hyp = decode_b(x_adv, mb)
        for i, (_, r) in enumerate(sub.iterrows()):
            ok = not np.isnan(eps[i])
            results.append(dict(utt_id=r["utt_id"], rung=RUNG, edit_type=e,
                checkpoint=CHECKPOINT, duration=r["duration"], source_text=r["text"],
                target_text=r["target_text"],
                eps50=float(eps[i]) if ok else None, censored=not ok,
                bracket_width=float(width[i]), snr_db=float(snrs[i]),
                decoded=(hyp[i] if ok else None),
                exact_match=(bool(hyp[i] == r["target_text"]) if ok else False),
                eps_lo=EPS_LO, eps_hi=EPS_HI, use_amp=USE_AMP, model_dir=MODEL_DIR))
        pd.DataFrame(results).to_csv(OUT, index=False)
        print(f"    batch {bi+1}/{len(batches)}  {dt/60:4.1f} min  "
              f"{int((~np.isnan(eps)).sum())}/{len(sub)} uncensored  {len(results)} rows")
    df = pd.DataFrame(results)
    nc = int(df.censored.astype(bool).sum()) if len(df) else 0
    summary.append(dict(edit_type=e, n=len(df), censored=nc,
                        asr=100.0*(len(df)-nc)/len(df) if len(df) else None,
                        complete=len(df) >= len(ev)))
    print(f"  {e}: {len(df)}/{len(ev)} measured, {nc} censored")

print(f"\nSESSION {(time.time()-SESSION_START)/3600:.2f} h")

## 6 · The paired ratio, against 9.34x at the 100-hour rung

In [ ]:
from math import comb
CEIL = float(EPS_HI)

def binom_two_sided(k, n):
    """Exact two-sided binomial test at p=0.5, method of small p-values."""
    if n == 0:
        return 1.0
    pmf = [comb(n, i) * 0.5 ** n for i in range(n + 1)]
    tol = pmf[k] * (1 + 1e-9)
    return min(1.0, sum(v for v in pmf if v <= tol))

def mcnemar(a, b):
    """a, b: dicts utt_id -> success bool, over the SAME utterances.
    Returns (p, n10, n01, n_common). Paired, exact."""
    ids = sorted(set(a) & set(b))
    n10 = sum(1 for u in ids if a[u] and not b[u])
    n01 = sum(1 for u in ids if b[u] and not a[u])
    return binom_two_sided(min(n10, n01), n10 + n01), n10, n01, len(ids)

def med(v):
    v = sorted(v); n = len(v)
    return None if not n else (v[n // 2] if n % 2 else 0.5 * (v[n//2 - 1] + v[n//2]))

def load_eps(path):
    """-> (utt_id -> success bool, utt_id -> imputed eps50)"""
    d = pd.read_csv(path)
    cens = d.censored.astype(str).str.lower().isin(["true", "1", "yes"])
    succ = {r.utt_id: (not c) for r, c in zip(d.itertuples(), cens)}
    eps = {r.utt_id: (CEIL if c else float(r.eps50))
           for r, c in zip(d.itertuples(), cens)}
    return succ, eps



In [ ]:
CEIL = 2e-3
def load_eps(p):
    d = pd.read_csv(p)
    c = d.censored.astype(str).str.lower().isin(["true","1","yes"])
    return ({r.utt_id: (not x) for r, x in zip(d.itertuples(), c)},
            {r.utt_id: (CEIL if x else float(r.eps50)) for r, x in zip(d.itertuples(), c)})

F = {}
for e in EDIT_TYPES:
    p = f"{WORK}/eps50_rung{RUNG}_{CHECKPOINT}_{e}_n{N_SHARED}.csv"
    if os.path.exists(p): F[e] = load_eps(p)

print("=" * 74); print(f"RQ1 x RQ2 --- the negation advantage at the {RUNG}-hour rung")
print("=" * 74)
REF = dict(t1_asr=100.0, t5_asr=36.8, ratio=9.34, rung="100 h final", n=38)
if len(F) < 2:
    print("  both edit types are needed for a paired ratio; only "
          f"{sorted(F)} finished. Publish and re-commit.")
else:
    s1, e1 = F["T1"]; s5, e5 = F["T5"]
    ids = sorted(set(e1) & set(e5))
    print(f"  paired on {len(ids)} utterances")
    a1 = 100.0*sum(s1[u] for u in ids)/len(ids)
    a5 = 100.0*sum(s5[u] for u in ids)/len(ids)
    print(f"\n  {'':10s}{'ASR@2e-3':>10s}{'censored':>10s}{'median eps50 (imputed)':>26s}")
    print(f"  {'T1':10s}{a1:9.1f}%{sum(1 for u in ids if not s1[u]):10d}"
          f"{med([e1[u] for u in ids]):26.4e}")
    print(f"  {'T5':10s}{a5:9.1f}%{sum(1 for u in ids if not s5[u]):10d}"
          f"{med([e5[u] for u in ids]):26.4e}")

    per = [e5[u]/e1[u] for u in ids]
    cheaper = sum(1 for u in ids if e1[u] < e5[u])
    ratio = med(per)
    p_mc, n10, n01, _ = mcnemar(s1, s5)
    print(f"\n  T1 cheaper on {cheaper}/{len(ids)} utterances")
    print(f"  PAIRED RATIO (median of per-utterance eps5/eps1) = {ratio:.2f}x")
    print(f"  McNemar on success, T1 vs T5: p = {p_mc:.3e}  ({n10} / {n01} discordant)")
    print(f"\n  reference at the {REF['rung']}, n={REF['n']}: "
          f"T1 {REF['t1_asr']:.0f}%, T5 {REF['t5_asr']:.1f}%, ratio {REF['ratio']}x")
    d = ratio - REF["ratio"]
    print(f"  difference: {d:+.2f}x")
    if abs(d) < 0.3 * REF["ratio"]:
        v = "SCALE-INVARIANT"
        print("\n  VERDICT: the negation advantage is essentially unchanged across a 100x")
        print("  difference in training data. That makes it a property of Bengali")
        print("  morphology rather than of model quality, which is a stronger claim than")
        print("  RQ2 currently makes. Put it in the RQ2 section and the abstract.")
    elif d < 0:
        v = "SHRINKS"
        print("\n  VERDICT: the advantage is SMALLER at low data. Part of the 9.34x needs a")
        print("  competent model, so RQ2 must be stated as conditional on accuracy.")
    else:
        v = "GROWS"
        print("\n  VERDICT: the advantage is LARGER at low data. Low-resource models are")
        print("  even more exposed to morphology-targeted edits -- this sharpens the")
        print("  thesis's own argument and belongs in the abstract.")
    print(f"\n  verdict = {v}")
    PROV.update(t1_asr=a1, t5_asr=a5, paired_ratio=ratio, mcnemar_p=p_mc,
                n_paired=len(ids), t1_cheaper=cheaper, verdict=v)

PROV["summary"] = summary
json.dump(PROV, open(f"{WORK}/_provenance.json", "w"), ensure_ascii=False, indent=2, default=str)
json.dump({"title": "eps50-rung1-t1t5-nb03", "id": "YOURUSERNAME/eps50-rung1-t1t5-nb03",
           "licenses": [{"name": "CC0-1.0"}]},
          open(f"{WORK}/dataset-metadata.json", "w"), indent=2)
print("\nfiles:", sorted(os.listdir(WORK)))
print("Publish as 'eps50-rung1-t1t5-nb03'. Check the list first.")

---

## Running this alongside NB-12

They are independent: different eval sets, different checkpoints, different output names. Confirm
your Kaggle account allows two concurrent GPU sessions, and watch the weekly quota — running both
burns it twice as fast.

NB-12 is the higher-priority one. If quota is tight, run NB-12 alone and do this next week.